In [1]:
import os
os.environ["SYCL_CACHE_PERSISTENT"] = "1"   # keep JIT-compiled GPU kernels on disk
os.environ.pop("ONEDNN_VERBOSE", None)       # silence the debug logging

In [2]:
import sys
import torch

# Checks if the current env uses cuda + nvidia gpu or xpu + intel gpu.

print(f"Python: {sys.executable}")
print(f"PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    device = torch.device("cuda:0")
    backend = "CUDA (NVIDIA)"
    device_name = torch.cuda.get_device_name(0)
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    device = torch.device("xpu:0")
    backend = "XPU (Intel)"
    device_name = torch.xpu.get_device_name(0)
else:
    device = torch.device("cpu")
    backend = "CPU"
    device_name = "CPU"

print(f"Backend: {backend}")
print(f"Device: {device_name}")
print(f"Device handle: {device}")

Python: /home/ofek/Desktop/UltimAI Solutions/ML/Hebrew Pronunciation Scorer/.venv/bin/python
PyTorch: 2.14.0+xpu
Backend: XPU (Intel)
Device: Intel(R) Arc(TM) Graphics
Device handle: xpu:0


In [3]:
# Cell 1.2 — one device for the whole notebook
def pick_device() -> str:
    if torch.cuda.is_available():
        return "cuda"                      # home PC (RTX)
    if hasattr(torch, "xpu") and torch.xpu.is_available():
        return "xpu"                       # laptop (Arc iGPU)
    return "cpu"

DEVICE = pick_device()
print("DEVICE =", DEVICE)

DEVICE = xpu


In [4]:
# Cell 1.3 — how fast is it? (matrix multiply benchmark)
# import torch, time, os
# print("ONEDNN_VERBOSE =", os.environ.get("ONEDNN_VERBOSE"))

# def step(name, fn):
#     t = time.perf_counter(); out = fn(); torch.xpu.synchronize()
#     print(f"{name:14} {time.perf_counter() - t:7.2f} s   {out}")

# step("alloc",        lambda: torch.ones(4, device="xpu").shape)
# step("elementwise",  lambda: (torch.ones(4, device="xpu") * 2).sum().item())
# a = torch.randn(256, 256, device="xpu")
# step("matmul #1",    lambda: (a @ a).sum().item())   # first call compiles the GPU kernel
# step("matmul #2",    lambda: (a @ a).sum().item())   # should be near-instant

In [5]:
# Cell 1.4 - test the phonemizer
from phonikud import phonemize
from phonikud_onnx import Phonikud

model = Phonikud("./phonikud-1.0.int8.onnx")
text = "שלום, מה שלומך?"
vocalized = model.add_diacritics(text)
phonemes = phonemize(vocalized)
print(phonemes)

/home/ofek/Desktop/UltimAI Solutions/ML/Hebrew Pronunciation Scorer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ʃalˈom, mˈa ʃlomχˈa?


In [7]:
# Cell 1.6 — decide the training precision from Cell 1.3's numbers
USE_BF16 = True    # keep True only if bf16 beat float32 on xpu; never use fp16 here

In [11]:
# Cell 2.1 — paths (the notebook lives in notebooks/, data one level up)
from pathlib import Path
ROOT = Path.cwd().parent
DATA = ROOT / "data"
(DATA / "raw").mkdir(parents=True, exist_ok=True)

In [8]:
# Cell 2.2 — the phoneme vocabulary
CONSONANTS = ["b","v","g","d","h","z","χ","t","j","k","l","m","n",
              "s","f","p","ts","ʁ","ʃ","ʔ","tʃ","dʒ","ʒ","w"]
VOWELS  = ["a","e","i","o","u"]
SPECIAL = ["<pad>", "<unk>", "|", "ˈ"]    # pad = CTC blank, | = word boundary, ˈ = stress
VOCAB = {s: i for i, s in enumerate(SPECIAL + CONSONANTS + VOWELS)}
ID2SYM = {i: s for s, i in VOCAB.items()}
print(len(VOCAB), "symbols")

33 symbols


In [9]:
# Cell 2.3 — peek at the dataset before assuming its column names
from datasets import load_dataset, Audio
ds = load_dataset("ivrit-ai/crowd-transcribe-v5", split="train", streaming=True, token=True)
first = next(iter(ds))
{k: type(v).__name__ for k, v in first.items()}

{'uuid': 'str',
 'audio': 'AudioDecoder',
 'orig_sentence': 'str',
 'sentence': 'str',
 'is_retranscribe': 'bool',
 'transcriber': 'int',
 'extra_data': 'dict'}

In [13]:
# Cell 2.4 — save 200 clips of 1–15 s as 16 kHz WAV files + a manifest
import soundfile as sf, pandas as pd

AUDIO_COL, TEXT_COL = "audio", "sentence"      # ← set from Cell 2.3's output
N_CLIPS = 200

ds = ds.cast_column(AUDIO_COL, Audio(sampling_rate=16_000))   # resample on the fly
rows = []
for ex in ds:
    a = ex[AUDIO_COL]
    dur = len(a["array"]) / a["sampling_rate"]
    if not 1 < dur < 15:                    # long clips blow up memory
        continue
    path = DATA / "raw" / f"{len(rows):05d}.wav"
    sf.write(path, a["array"], 16_000)
    rows.append({"path": str(path), "text": ex[TEXT_COL], "duration": dur})
    if len(rows) >= N_CLIPS:
        break

manifest = pd.DataFrame(rows)
manifest.to_parquet(DATA / "raw" / "manifest.parquet")
manifest.head()

,path,text,duration
0,/home/ofek/Desktop/UltimAI Solutions/ML/data/r...,"לא, זה לא אני קטונתי כן אבל אני במקומו הייתי ה...",6.335750
1,/home/ofek/Desktop/UltimAI Solutions/ML/data/r...,‫יותר סביר שהדברים האלה ‫נאמרו לעומת דברים אחרים.,3.619063
2,/home/ofek/Desktop/UltimAI Solutions/ML/data/r...,"ומחכים שגנב יגיע, ואז אתם תגידו לו, מה אתה עוש...",5.526000
3,/home/ofek/Desktop/UltimAI Solutions/ML/data/r...,"‫אז זה לא מעניין, ‫אנחנו נותנים את הקונטקסט הזה.",2.125000
4,/home/ofek/Desktop/UltimAI Solutions/ML/data/r...,‫אברהם מניח שבמקרה ‫של התפלגות נורמלית...,2.341000


In [14]:
# Cell 2.4b — sanity checks on what was saved
from IPython.display import Audio as Play, display
print(len(manifest), "clips,", round(manifest.duration.sum() / 60, 1), "minutes total")
print(manifest.duration.describe().round(2))

row = manifest.iloc[0]
print(row.text)
display(Play(filename=row.path))

200 clips, 15.2 minutes total
count    200.00
mean       4.56
std        2.67
min        2.05
25%        2.65
50%        3.60
75%        5.53
max       14.96
Name: duration, dtype: float64
לא, זה לא אני קטונתי כן אבל אני במקומו הייתי הולך על מהלך אמרתי לך ללכת כזה לאקדמיה של


In [ ]:
# cell 2.5 - convert a string to a list of phonemes using phonikud and phonemizer
def to_phonemes(text: str) -> list[str]:
    """Convert a string to a list of phonemes."""
    vocalized = model.add_diacritics(text)
    phonemes = phonemize(vocalized)
    return phonemes.split()

to_phonemes("קוראים לי אופק") # Test the function with a sample string

['koʁʔˈim', 'lˈi', 'ʔˈofek']

In [18]:
# Cell 2.6 — label every clip, then eyeball a few
manifest["sentence"] = (
    manifest.text
    .str.replace(r"[^א-ת\s]", "", regex=True)
    .str.split()
    .str.join(" ")
)
manifest["phonemes"] = manifest.text.map(lambda t: " ".join(to_phonemes(t)))
manifest = manifest[manifest.phonemes.str.len() > 0]
manifest.to_parquet(DATA / "labels.parquet")
manifest.sample(5, random_state=0)[["text", "phonemes"]]

,text,phonemes
18,"שם פנטסטי אגב, הסכין אורי, סחטיין עליך.","ʃˈam fantˈasti ʔaɡˈav, hasakˈin ʔuʁˈi, saχtˈen..."
170,האם אתה צריך לעשות אלרט שהCPU שלך גבוה?,haʔˈim ʔatˈa tsaʁˈiχ laʔasˈot ʔˈelʁt ʃeˈa ʃelχ...
107,"אני לא יכולה לדבר עם אייטיונס בעצמי, אני צריכה...",ʔanˈi lˈo jeχolˈa ledabˈeʁ ʔˈim ʔejtjˈons beʔa...
98,"אוקיי אז אתה צודק, 42% מהמישיבים הצביעו מהמנזר.","ʔokˈej ʔˈaz ʔatˈa tsodˈek, ʔaʁbaʔˈim veʃtˈajim..."
177,ורק אחרי מספר שבועות הוא יתעורר.,veʁˈak ʔaχaʁˈej mispˈaʁ ʃavuʔˈot hˈu jitʔoʁˈeʁ.


In [19]:
# Cell 3.1 — hold out clean clips BEFORE training (stand-in for the 300 verified ones)
labels = pd.read_parquet(DATA / "labels.parquet").sample(frac=1, random_state=0)  # shuffle
gold_clean = labels.iloc[:20]
train_pool = labels.iloc[20:]
(DATA / "gold").mkdir(exist_ok=True)
gold_clean.to_parquet(DATA / "gold" / "clean.parquet")
train_pool.to_parquet(DATA / "train_pool.parquet")
len(gold_clean), len(train_pool)

(20, 180)